# Booking Performance Analysis

Objective:
Analyze booking performance, customer behavior, property performance, destination popularity, and host performance using the booking_details analytical view.

In [0]:
%sql
USE CATALOG wanderbricks_analytics;
USE SCHEMA analytics;

---
## 1. Business Overview

This section establishes the overall scale and performance of booking activity. It examines booking volume, booking value, trends over time, and the current distribution of booking statuses.

In [0]:
%sql
SELECT COUNT(booking_id) AS total_booking, 
    ROUND(SUM(total_amount), 2) AS total_booking_value, 
    ROUND(AVG(total_amount), 2) AS avg_booking_value
FROM booking_details

In [0]:
%sql
SELECT 
    year(to_timestamp(booking_created_at)) AS Year,
    month(to_timestamp(booking_created_at)) AS Month,
    COUNT(booking_id) AS total_booking
FROM booking_details
GROUP BY Year, Month
ORDER BY Year, Month

In [0]:
%sql
SELECT 
    year(to_timestamp(booking_created_at)) AS year,
    month(to_timestamp(booking_created_at)) AS Month,
    ROUND(SUM(total_amount), 2) AS total_amount
FROM booking_details
GROUP BY Year, Month
ORDER BY Year, Month

In [0]:
%sql
SELECT 
    status,
    COUNT(booking_id) AS booking_count,
    ROUND(
        COUNT(booking_id) * 100.0
        / SUM(COUNT(booking_id)) OVER (),
        2
    ) AS booking_percentage
FROM booking_details
GROUP BY status

### Key Findings

- 72,247 bookings generated approximately 40.0M in total recorded booking value, with an average booking value of 553.77.
- Booking activity increased substantially over the available period, from fewer than 500 monthly bookings in late 2023 to over 10,000 by July 2025.
- Monthly booking value generally followed the same upward pattern as booking volume, reaching approximately 5.7M in July 2025.
- Pending bookings make up the largest share of bookings (43.70%), while completed bookings account for only 10.30%.
- The relatively large proportion of pending and cancelled bookings suggests that booking status should be considered when evaluating realized booking performance.

---
## 2. Customer Analysis

This section examines customer composition and geographic booking patterns to understand which customer groups and markets contribute most to booking activity and booking value.

In [0]:
%sql
SELECT user_type,
    COUNT(booking_id) AS booking_count,
    ROUND(COUNT(booking_id) * 100 / SUM(COUNT(booking_id)) OVER (), 2) AS booking_percentage
FROM booking_details
GROUP BY user_type
ORDER BY booking_count DESC

In [0]:
%sql
SELECT user_country,
    COUNT(booking_id) AS booking_count
FROM booking_details
GROUP BY user_country
ORDER BY booking_count DESC
LIMIT 10

In [0]:
%sql
SELECT user_country,
    ROUND(AVG(total_amount), 2) AS avg_booking_value,
    COUNT(booking_id) AS booking_count
FROM booking_details
GROUP BY user_country
HAVING COUNT(booking_id) >= 100
ORDER BY avg_booking_value DESC
LIMIT 10

### Key Findings

- Booking activity is almost evenly split between business and individual customers, with business customers accounting for 50.04% and individual customers 49.96%.
- India and China are the largest customer markets by booking volume, with 13,435 and 13,081 bookings respectively, far exceeding the next-largest market, the United States with 3,208 bookings.
- Among countries with at least 100 bookings, Sri Lanka has the highest average booking value at 595.19, followed by Cuba (590.73) and Burundi (590.69).
- The markets with the highest booking volumes are not necessarily the markets with the highest average booking values, suggesting that booking volume and booking value represent different aspects of customer performance.

---
## 3. Property & Destination Analysis

This section examines how property types and destinations contribute to booking volume, booking value, and customer stay behavior.

In [0]:
%sql
SELECT property_type, 
    ROUND(SUM(total_amount), 2) AS booking_value,
    COUNT(booking_id) AS booking_count
FROM booking_details
GROUP BY property_type
ORDER BY booking_value DESC

In [0]:
%sql
SELECT property_type, 
    ROUND(AVG(stay_length), 2) AS average_stay,
    COUNT(booking_id) AS booking_count
FROM booking_details
GROUP BY property_type
ORDER BY average_stay DESC

In [0]:
%sql
SELECT destination_name,
    ROUND(COUNT(booking_id), 2) AS booking_count
FROM booking_details
GROUP BY destination_name
ORDER BY booking_count DESC
LIMIT 10

In [0]:
%sql
SELECT destination_name, 
    ROUND(SUM(total_amount), 2) AS booking_value,
    COUNT(booking_id) AS booking_count
FROM booking_details
GROUP BY destination_name
ORDER BY booking_value DESC
LIMIT 10

### Key Findings

- Urban Year-Round properties generate the highest booking value at approximately 20.4M, followed by Summer Getaway properties at approximately 15.7M. Ski Resorts contribute relatively little booking value in comparison.
- Average stay length varies only slightly across property types, ranging from 3.69 days for Summer Getaways to 3.85 days for Ski Resorts.
- Phuket is the most popular destination by booking volume, with 7,292 bookings, followed by Gold Coast (6,507) and Mallorca (6,487).
- The top destinations by booking volume also rank highly in total booking value, with Phuket generating approximately 4.1M.
- The strong contribution from Urban Year-Round and Summer Getaway properties suggests these categories are the primary drivers of booking value in the dataset.

---
## 4. Host Analysis

This section examines host performance and evaluates whether host characteristics such as revenue contribution, ratings, and verification status are associated with booking performance.


In [0]:
%sql
SELECT host_id, COUNT(booking_id), ROUND(SUM(total_amount), 2) AS booking_value
FROM booking_details
GROUP BY host_id
ORDER BY booking_value DESC
LIMIT 10

In [0]:
%sql
SELECT 
    CASE
        WHEN host_rating < 2.5 THEN '< 2.5'
        WHEN host_rating < 3.0 THEN '2.5 - 2.99'
        WHEN host_rating < 3.5 THEN '3.0 - 3.49'
        WHEN host_rating < 4.0 THEN '3.5 - 3.99'
        WHEN host_rating < 4.5 THEN '4.0 - 4.49'
        ELSE '4.5+'
    END AS rating_band,
    ROUND(SUM(total_amount), 2) AS booking_value,
    COUNT(booking_id) AS booking_count,
    ROUND(AVG(total_amount), 2) AS avg_booking_value
FROM booking_details
GROUP BY 
    CASE
        WHEN host_rating < 2.5 THEN '< 2.5'
        WHEN host_rating < 3.0 THEN '2.5 - 2.99'
        WHEN host_rating < 3.5 THEN '3.0 - 3.49'
        WHEN host_rating < 4.0 THEN '3.5 - 3.99'
        WHEN host_rating < 4.5 THEN '4.0 - 4.49'
        ELSE '4.5+'
    END
ORDER BY rating_band;

In [0]:
%sql
SELECT is_verified, COUNT(booking_id) AS booking_count,  ROUND(SUM(total_amount), 2) AS booking_value,  ROUND(AVG(total_amount), 2) AS avg_booking_value
FROM booking_details
GROUP BY is_verified
ORDER BY is_verified

### Key Findings

- Host rating does not show a clear relationship with average booking value across the rating bands.
- Hosts rated below 2.5 have the highest average booking value at $562.80, while hosts rated 4.5+ have the lowest at $549.85.
- The 3.5–3.99 rating band generates the highest total booking value ($13.05M), largely reflecting its higher booking volume.
- Overall, the analysis does not provide strong evidence that higher host ratings correspond to higher booking value.

---
## 5. Marketing Analysis

This section examines property page-view activity to understand how users discover and interact with properties. It focuses on device usage, traffic sources, property types, and destinations receiving the most attention.

In [0]:
%sql
SELECT device_type,
    COUNT(view_id) AS view_count, 
    ROUND(COUNT(view_id)  * 100 /SUM(COUNT(view_id)) OVER(), 2) AS view_percentage
FROM property_marketing
GROUP BY device_type
ORDER BY view_count DESC

In [0]:
%sql
SELECT referrer, COUNT(view_id) AS view_count,
    ROUND( COUNT(view_id) * 100/SUM(COUNT(view_id)) OVER(), 2) AS view_percentage
FROM property_marketing
GROUP BY referrer
ORDER BY view_count DESC

In [0]:
%sql
SELECT property_type, COUNT(view_id) AS view_count,
    ROUND( COUNT(view_id) * 100/SUM(COUNT(view_id)) OVER(), 2) AS view_percentage
FROM property_marketing
GROUP BY property_type
ORDER BY view_count DESC

In [0]:
%sql
SELECT 
    d.destination AS destination_name,
    COUNT(pm.view_id) AS view_count,
    ROUND(
        COUNT(pm.view_id) * 100.0 / SUM(COUNT(pm.view_id)) OVER (),
        2
    ) AS view_percentage
FROM property_marketing pm
JOIN samples.wanderbricks.destinations d
    ON pm.destination_id = d.destination_id
GROUP BY d.destination
ORDER BY view_count DESC
LIMIT 10;

### Key Findings

- Property views are almost evenly distributed across devices, with tablets generating the most views (33.42%), followed closely by mobile (33.33%) and desktop (33.25%). This suggests that the platform receives substantial traffic across all three device types.
- Traffic sources are also remarkably balanced. Google generates the most views (25.03%), followed closely by email (25.01%), ads (24.98%), and direct traffic (24.97%). No single traffic source dominates.
- Urban Year-Round properties receive the most attention, accounting for 50.84% of all property views, followed by Summer Getaway properties at 39.08%. Historical Place and Ski Resort properties receive substantially fewer views.
- Phuket receives the highest number of property views among destinations, with 49,399 views (9.88%), followed by Mallorca and Gold Coast. The top destinations therefore represent a significant share of overall property interest.

# Final Business Summary

## Key Findings

- The platform recorded 72,247 bookings with approximately $40.0M in total booking value, while booking activity increased substantially over the observed period.
- Business and individual customers contribute almost equally to booking volume, while India and China are the largest customer markets.
- Urban Year-Round and Summer Getaway properties are the primary contributors to booking value and property views.
- Phuket is the leading destination in both booking activity and property views.
- Host rating does not show a clear relationship with average booking value.
- Marketing traffic is broadly distributed across devices and traffic sources, with no single channel or device dominating overall views.

## Recommendations

- Investigate the high proportion of pending and cancelled bookings to identify opportunities to improve booking completion.
- Prioritize high-performing property types and destinations when evaluating inventory and marketing opportunities.
- Monitor customer markets separately for booking volume and average booking value, as these metrics identify different types of market performance.
- Avoid relying on host rating alone when evaluating revenue potential, and consider additional factors such as property type, destination, and booking volume.